# 

Practical Application III: Comparing Classifiers

**Overview**: In this practical application, your goal is to compare the performance of the classifiers we encountered in this section, namely K Nearest Neighbor, Logistic Regression, Decision Trees, and Support Vector Machines.  We will utilize a dataset related to marketing bank products over the telephone.  

### Getting Started

Our dataset comes from the UCI Machine Learning repository [link](https://archive.ics.uci.edu/ml/datasets/bank+marketing).  The data is from a Portugese banking institution and is a collection of the results of multiple marketing campaigns.  We will make use of the article accompanying the dataset [here](CRISP-DM-BANK.pdf) for more information on the data and features.



### Problem 1: Understanding the Data

To gain a better understanding of the data, please read the information provided in the UCI link above, and examine the **Materials and Methods** section of the paper.  How many marketing campaigns does this data represent?

In [1]:
import pandas as pd

bank_data = pd.read_csv("bank-additional-full.csv", sep=';')

# Remove leakage feature
bank_data = bank_data.drop(columns=['duration'])

# Encode target
bank_data['y'] = bank_data['y'].map({'yes': 1, 'no': 0})

# Fix pdays
bank_data['pdays'] = bank_data['pdays'].replace(999, -1)

bank_data.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,1,-1,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,1,-1,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0
2,37,services,married,high.school,no,yes,no,telephone,may,mon,1,-1,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,1,-1,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0
4,56,services,married,high.school,no,no,yes,telephone,may,mon,1,-1,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0


### Problem 2: Read in the Data

Use pandas to read in the dataset `bank-additional-full.csv` and assign to a meaningful variable name.

In [2]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split

# Features
features = ['age', 'job', 'marital', 'education', 'default', 'housing', 'loan']

X = bank_data[features]
y = bank_data['y']

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Preprocessing
cat_cols = X.select_dtypes(include='object').columns
num_cols = X.select_dtypes(exclude='object').columns

preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
    ('num', 'passthrough', num_cols)
])

# Pipeline
pipe_log = Pipeline([
    ('prep', preprocessor),
    ('model', LogisticRegression(max_iter=1000))
])

# Fit
pipe_log.fit(X_train, y_train)

# Predict
log_pred = pipe_log.predict(X_test)

### Problem 3: Understanding the Features


Examine the data description below, and determine if any of the features are missing values or need to be coerced to a different data type.


```
Input variables:
# bank client data:
1 - age (numeric)
2 - job : type of job (categorical: 'admin.','blue-collar','entrepreneur','housemaid','management','retired','self-employed','services','student','technician','unemployed','unknown')
3 - marital : marital status (categorical: 'divorced','married','single','unknown'; note: 'divorced' means divorced or widowed)
4 - education (categorical: 'basic.4y','basic.6y','basic.9y','high.school','illiterate','professional.course','university.degree','unknown')
5 - default: has credit in default? (categorical: 'no','yes','unknown')
6 - housing: has housing loan? (categorical: 'no','yes','unknown')
7 - loan: has personal loan? (categorical: 'no','yes','unknown')
# related with the last contact of the current campaign:
8 - contact: contact communication type (categorical: 'cellular','telephone')
9 - month: last contact month of year (categorical: 'jan', 'feb', 'mar', ..., 'nov', 'dec')
10 - day_of_week: last contact day of the week (categorical: 'mon','tue','wed','thu','fri')
11 - duration: last contact duration, in seconds (numeric). Important note: this attribute highly affects the output target (e.g., if duration=0 then y='no'). Yet, the duration is not known before a call is performed. Also, after the end of the call y is obviously known. Thus, this input should only be included for benchmark purposes and should be discarded if the intention is to have a realistic predictive model.
# other attributes:
12 - campaign: number of contacts performed during this campaign and for this client (numeric, includes last contact)
13 - pdays: number of days that passed by after the client was last contacted from a previous campaign (numeric; 999 means client was not previously contacted)
14 - previous: number of contacts performed before this campaign and for this client (numeric)
15 - poutcome: outcome of the previous marketing campaign (categorical: 'failure','nonexistent','success')
# social and economic context attributes
16 - emp.var.rate: employment variation rate - quarterly indicator (numeric)
17 - cons.price.idx: consumer price index - monthly indicator (numeric)
18 - cons.conf.idx: consumer confidence index - monthly indicator (numeric)
19 - euribor3m: euribor 3 month rate - daily indicator (numeric)
20 - nr.employed: number of employees - quarterly indicator (numeric)

Output variable (desired target):
21 - y - has the client subscribed a term deposit? (binary: 'yes','no')
```



### Problem 4: Understanding the Task

After examining the description and data, your goal now is to clearly state the *Business Objective* of the task.  State the objective below.

In [ ]:
#df.info()

### Problem 5: Engineering Features

Now that you understand your business objective, we will build a basic model to get started.  Before we can do this, we must work to encode the data.  Using just the bank information features, prepare the features and target column for modeling with appropriate encoding and transformations.

### Problem 6: Train/Test Split

With your data prepared, split it into a train and test set.

In [3]:
from sklearn.model_selection import train_test_split

# 1. Split data first
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(y_train.value_counts(normalize=True))

y
0    0.887344
1    0.112656
Name: proportion, dtype: float64


### Problem 7: A Baseline Model

Before we build our first model, we want to establish a baseline.  What is the baseline performance that our classifier should aim to beat?

In [ ]:
y.value_counts(normalize=True)

### Problem 8: A Simple Model

Use Logistic Regression to build a basic model on your data.  

In [4]:
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
X_train_encoded = preprocessor.fit_transform(X_train)
X_test_encoded = preprocessor.transform(X_test)
#  SVM
svm = LinearSVC(class_weight='balanced')
print("Training SVM...")
svm.fit(X_train_encoded, y_train)
print("SVM done")
svm_pred = svm.predict(X_test_encoded)

#  Decision Tree
tree = DecisionTreeClassifier(random_state=42, class_weight='balanced')
print("Training Tree...")
tree.fit(X_train_encoded, y_train)
print("Tree done")
tree_pred = tree.predict(X_test_encoded)
#  KNN
knn = KNeighborsClassifier(n_neighbors=5)
print("Training KNN...")
knn.fit(X_train_encoded, y_train)
print("KNN done")
knn_pred = knn.predict(X_test_encoded)

#  Logistic Regression
log_model = LogisticRegression(max_iter=1000, class_weight='balanced')
print("Training Logistic...")
log_model.fit(X_train_encoded, y_train)
log_pred = log_model.predict(X_test_encoded)
print("Logistic done")



Training SVM...
SVM done
Training Tree...
Tree done
Training KNN...
KNN done
Training Logistic...
Logistic done


### Problem 9: Score the Model

What is the accuracy of your model?

In [5]:
from sklearn.metrics import accuracy_score, f1_score, recall_score

def evaluate(name, y_test, y_pred):
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "F1 Score": f1_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred)
    }

### Problem 10: Model Comparisons

Now, we aim to compare the performance of the Logistic Regression model to our KNN algorithm, Decision Tree, and SVM models.  Using the default settings for each of the models, fit and score each.  Also, be sure to compare the fit time of each of the models.  Present your findings in a `DataFrame` similar to that below:

| Model | Train Time | Train Accuracy | Test Accuracy |
| ----- | ---------- | -------------  | -----------   |
|     |    |.     |.     |

In [6]:
results = []

results.append(evaluate("Logistic Regression", y_test, log_pred))
results.append(evaluate("KNN", y_test, knn_pred))
results.append(evaluate("Decision Tree", y_test, tree_pred))
results.append(evaluate("SVM", y_test, svm_pred))

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="F1 Score", ascending=False)
print(results_df)

                 Model  Accuracy  F1 Score    Recall
0  Logistic Regression  0.583758  0.253754  0.628233
3                  SVM  0.583880  0.251855  0.621767
2        Decision Tree  0.683661  0.231722  0.423491
1                  KNN  0.876912  0.119792  0.074353


### Problem 11: Improving the Model

Now that we have some basic models on the board, we want to try to improve these.  Below, we list a few things to explore in this pursuit.


- Hyperparameter tuning and grid search.  All of our models have additional hyperparameters to tune and explore.  For example the number of neighbors in KNN or the maximum depth of a Decision Tree.  
- Adjust your performance metric

In [7]:
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

from sklearn.metrics import accuracy_score, f1_score, recall_score

# -----------------------------
# 1. Load Data
# -----------------------------
bank_data = pd.read_csv("bank-additional-full.csv", sep=';')

# -----------------------------
# 2. Clean Data
# -----------------------------
bank_data = bank_data.drop(columns=['duration'])
bank_data['y'] = bank_data['y'].map({'yes': 1, 'no': 0})
bank_data['pdays'] = bank_data['pdays'].replace(999, -1)

# -----------------------------
# 3. Features / Target
# -----------------------------
features = ['age', 'job', 'marital', 'education', 'default', 'housing', 'loan']

X = bank_data[features]
y = bank_data['y']

# -----------------------------
# 4. Baseline
# -----------------------------
baseline_accuracy = y.value_counts(normalize=True).max()
print("Baseline Accuracy:", baseline_accuracy)

# -----------------------------
# 5. Preprocessing
# -----------------------------
cat_cols = X.select_dtypes(include='object').columns
num_cols = X.select_dtypes(exclude='object').columns

preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
    ('num', 'passthrough', num_cols)
])

# -----------------------------
# 6. Train-Test Split
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# -----------------------------
# 7. Encode Data
# -----------------------------
X_train_encoded = preprocessor.fit_transform(X_train)
X_test_encoded = preprocessor.transform(X_test)

print("Shapes:")
print(X_train_encoded.shape)
print(X_test_encoded.shape)

# -----------------------------
# 8. Models + Grid Search (FAST VERSION)
# -----------------------------

# Logistic Regression
grid_log = GridSearchCV(
    LogisticRegression(max_iter=1000),
    {'C': [1], 'class_weight': ['balanced']},
    cv=3,
    scoring='f1',
    n_jobs=1,
    verbose=0
)
grid_log.fit(X_train_encoded, y_train)
log_pred = grid_log.predict(X_test_encoded)

# KNN
grid_knn = GridSearchCV(
    KNeighborsClassifier(),
    {'n_neighbors': [5], 'weights': ['uniform']},
    cv=3,
    scoring='f1',
    n_jobs=1,
    verbose=0
)
grid_knn.fit(X_train_encoded, y_train)
knn_pred = grid_knn.predict(X_test_encoded)

# Decision Tree
grid_tree = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    {'max_depth': [5], 'class_weight': ['balanced']},
    cv=3,
    scoring='f1',
    n_jobs=1,
    verbose=0
)
grid_tree.fit(X_train_encoded, y_train)
tree_pred = grid_tree.predict(X_test_encoded)

# SVM (linear only = fast)
grid_svm = GridSearchCV(
    SVC(),
    {'C': [1], 'kernel': ['linear'], 'class_weight': ['balanced']},
    cv=3,
    scoring='f1',
    n_jobs=1,
    verbose=0
)
grid_svm.fit(X_train_encoded, y_train)
svm_pred = grid_svm.predict(X_test_encoded)

# -----------------------------
# 9. Evaluation Function
# -----------------------------
def evaluate(name, y_test, y_pred):
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "F1 Score": f1_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred)
    }

# -----------------------------
# 10. Results Table
# -----------------------------
results = []

results.append(evaluate("Logistic (Tuned)", y_test, log_pred))
results.append(evaluate("KNN (Tuned)", y_test, knn_pred))
results.append(evaluate("Decision Tree (Tuned)", y_test, tree_pred))
results.append(evaluate("SVM (Tuned)", y_test, svm_pred))

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="F1 Score", ascending=False)

print("\nFinal Results:")
print(results_df)

# -----------------------------
# 11. Best Model
# -----------------------------
best_model = results_df.iloc[0]["Model"]
print("\nBest Model:", best_model)

# -----------------------------
# 12. Best Parameters
# -----------------------------
print("\nBest Logistic Params:", grid_log.best_params_)
print("Best KNN Params:", grid_knn.best_params_)
print("Best Tree Params:", grid_tree.best_params_)
print("Best SVM Params:", grid_svm.best_params_)
print("DONE")

Baseline Accuracy: 0.8873458288821987
Shapes:
(32950, 34)
(8238, 34)

Final Results:
                   Model  Accuracy  F1 Score    Recall
2  Decision Tree (Tuned)  0.696650  0.262179  0.478448
0       Logistic (Tuned)  0.583758  0.253754  0.628233
3            SVM (Tuned)  0.606943  0.250810  0.584052
1            KNN (Tuned)  0.876912  0.119792  0.074353

Best Model: Decision Tree (Tuned)

Best Logistic Params: {'C': 1, 'class_weight': 'balanced'}
Best KNN Params: {'n_neighbors': 5, 'weights': 'uniform'}
Best Tree Params: {'class_weight': 'balanced', 'max_depth': 5}
Best SVM Params: {'C': 1, 'class_weight': 'balanced', 'kernel': 'linear'}
DONE


In [8]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC

from sklearn.metrics import accuracy_score, f1_score, recall_score, ConfusionMatrixDisplay

# -----------------------------
# 1. Load Data
# -----------------------------
bank_data = pd.read_csv("bank-additional-full.csv", sep=';')

print("STEP 1: Data loaded")
print(bank_data.shape)

bank_data = bank_data.drop(columns=['duration'])
bank_data['y'] = bank_data['y'].map({'yes': 1, 'no': 0})
bank_data['pdays'] = bank_data['pdays'].replace(999, -1)

# -----------------------------
# 2. Features / Target
# -----------------------------
features = ['age', 'job', 'marital', 'education', 'default', 'housing', 'loan']

X = bank_data[features]
y = bank_data['y']

print("STEP 2: Features created")
print(X.shape, y.shape)

# -----------------------------
# 3. Preprocessing (DEFINE FIRST)
# -----------------------------
cat_cols = X.select_dtypes(include='object').columns
num_cols = X.select_dtypes(exclude='object').columns

print("STEP 3: Columns identified")
print("Categorical:", list(cat_cols))
print("Numerical:", list(num_cols))

preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
    ('num', StandardScaler(), num_cols)
])

# -----------------------------
# 4. Train/Test Split
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("STEP 4: Split done")
print(X_train.shape, X_test.shape)

# -----------------------------
# 5. Encoding
# -----------------------------
X_train_enc = preprocessor.fit_transform(X_train)
X_test_enc = preprocessor.transform(X_test)

print("STEP 5: Encoding done")
print(X_train_enc.shape, X_test_enc.shape)

# -----------------------------
# 6. Evaluation function
# -----------------------------
def evaluate(name, y_true, y_pred):
    print(f"{name} evaluated")   # DEBUG LINE
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "F1 Score": f1_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred)
    }

results = []

# -----------------------------
# 7. Logistic Regression
# -----------------------------
print("Training Logistic...")
log_grid = GridSearchCV(
    LogisticRegression(max_iter=1000),
    {'C': [1], 'class_weight': ['balanced']},
    cv=3,
    scoring='f1',
    n_jobs=-1
)

log_grid.fit(X_train_enc, y_train)
log_pred = log_grid.predict(X_test_enc)
results.append(evaluate("Logistic", y_test, log_pred))

# -----------------------------
# 8. KNN
# -----------------------------
print("Training KNN...")
knn_grid = GridSearchCV(
    KNeighborsClassifier(),
    {'n_neighbors': [5], 'weights': ['uniform']},
    cv=3,
    scoring='f1',
    n_jobs=-1
)

knn_grid.fit(X_train_enc, y_train)
knn_pred = knn_grid.predict(X_test_enc)
results.append(evaluate("KNN", y_test, knn_pred))

# -----------------------------
# 9. Decision Tree
# -----------------------------
print("Training Tree...")
tree_grid = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    {'max_depth': [5], 'class_weight': ['balanced']},
    cv=3,
    scoring='f1',
    n_jobs=-1
)

tree_grid.fit(X_train_enc, y_train)
tree_pred = tree_grid.predict(X_test_enc)
results.append(evaluate("Decision Tree", y_test, tree_pred))

# -----------------------------
# 10. SVM
# -----------------------------
print("Training SVM...")
svm_grid = GridSearchCV(
    LinearSVC(),
    {'C': [1]},
    cv=3,
    scoring='f1',
    n_jobs=-1
)

svm_grid.fit(X_train_enc, y_train)
svm_pred = svm_grid.predict(X_test_enc)
results.append(evaluate("SVM", y_test, svm_pred))

# -----------------------------
# 11. Results
# -----------------------------
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="F1 Score", ascending=False)

print("\nFINAL RESULTS:")
print(results_df)

best_model = results_df.iloc[0]["Model"]
print("\nBest Model:", best_model)

STEP 1: Data loaded
(41188, 21)
STEP 2: Features created
(41188, 7) (41188,)
STEP 3: Columns identified
Categorical: ['job', 'marital', 'education', 'default', 'housing', 'loan']
Numerical: ['age']
STEP 4: Split done
(32950, 7) (8238, 7)
STEP 5: Encoding done
(32950, 34) (8238, 34)
Training Logistic...
Logistic evaluated
Training KNN...
KNN evaluated
Training Tree...
Decision Tree evaluated
Training SVM...
SVM evaluated

FINAL RESULTS:
           Model  Accuracy  F1 Score    Recall
2  Decision Tree  0.696650  0.262179  0.478448
0       Logistic  0.584365  0.252075  0.621767
1            KNN  0.880189  0.122667  0.074353
3            SVM  0.887351  0.000000  0.000000

Best Model: Decision Tree


##### Questions